# Claude API, System Prompts, Prompt Engineering & Prompt Evaluation
### Explained very simply, start to finish, with ONE example all the way through

**Our one example for the whole notebook:**

> We run a small online shop. Customers send us messages. We want Claude to read each message and give us back tidy data: what the message is about, how urgent it is, and a one line summary.

We will do this in 6 steps:

1. Talk to Claude with the API
2. Add a **system prompt**
3. Write a **first draft prompt** (and watch it be a bit rubbish)
4. Build an **evaluation**: test cases + a grader
5. **Prompt engineering**: improve the prompt
6. Re-run the evaluation and compare scores

Think of it like cooking: the prompt is the recipe, the evaluation is the taste test, and prompt engineering is adjusting the recipe after tasting.


## Step 0. Setup

Install the library and set your API key.

Get an API key from the Claude Console (console.anthropic.com). Once logged in, look at the **left sidebar**, click **API Keys**, then the **Create Key** button at the top right of that page. Copy the key immediately, it is only shown once.

Never paste your key directly into a notebook you might share. Use an environment variable instead.


In [1]:
%pip install anthropic --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from anthropic import Anthropic

# Set this in your terminal before starting Jupyter:
#   Mac/Linux:  export ANTHROPIC_API_KEY="sk-ant-..."
#   Windows:    setx ANTHROPIC_API_KEY "sk-ant-..."
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Model IDs. Confirm current ones at docs.claude.com under "Models overview".
BIG_MODEL = "claude-sonnet-5"              # the model doing the real work
SMALL_MODEL = "claude-haiku-4-5-20251001"  # cheap + fast, for side jobs
print("ready")

ready


## Step 1. The simplest possible API call

Three things you always send:

1. `model`: which Claude you want
2. `max_tokens`: the longest reply you will allow
3. `messages`: the conversation so far, as a list

Claude has **no memory between calls**. Every call is a fresh start. If you want it to remember, you resend the history yourself.


In [3]:
response = client.messages.create(
    model=BIG_MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": "Say hello in one short sentence."}
    ],
)

print(response.content[0].text)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CdDk5rnGGs7NZqyDJkLrf'}

**What came back?**

`response.content` is a *list* of blocks. For plain text answers there is one block, and the text lives at `response.content[0].text`.

You also get `response.usage.input_tokens` and `response.usage.output_tokens`, which is how you are billed.


In [ ]:
print("input tokens :", response.usage.input_tokens)
print("output tokens:", response.usage.output_tokens)

## Step 2. System prompts

A **system prompt** is a separate instruction that sets Claude's role and rules. It is not part of the conversation, it sits above it.

Simple way to think about it:

- **System prompt** = the job description. Who you are, what rules you follow, what format you output.
- **User message** = the actual task right now. The one ticket to classify.

Why bother? It keeps your instructions in one stable place, and Claude weights it strongly. It also means your user message can just be the raw customer text, which is much cleaner.

In the Python library it is the `system=` argument, **not** a message.


In [ ]:
SYSTEM_V1 = "You are a helpful assistant that sorts customer support messages for an online shop."

TICKET = "hi i ordered the blue trainers 9 days ago, tracking hasnt moved since tuesday and i need them for a wedding on saturday. can someone actually look at this please"

response = client.messages.create(
    model=BIG_MODEL,
    max_tokens=500,
    system=SYSTEM_V1,
    messages=[{"role": "user", "content": TICKET}],
)

print(response.content[0].text)

## Step 3. Our first draft prompt (deliberately weak)

Now let's ask for the actual job: category, urgency, summary.

Watch what goes wrong. This is the "I tested it once and it worked" trap from the quiz.


In [ ]:
PROMPT_V1 = """You are a support assistant for an online shop.
Read the customer message and tell me the category, the urgency, and a summary."""

def classify(ticket, system_prompt, model=BIG_MODEL):
    r = client.messages.create(
        model=model,
        max_tokens=500,
        system=system_prompt,
        messages=[{"role": "user", "content": ticket}],
    )
    return r.content[0].text

print(classify(TICKET, PROMPT_V1))

**Problems you will probably see:**

- It writes a friendly paragraph instead of data
- Categories are invented on the spot, so they differ every time
- Urgency might be "high", or "very urgent", or "8/10"
- You cannot feed this into a spreadsheet or a database

The prompt is not *wrong*, it is just **vague**. Vague prompts give inconsistent answers. That is the whole reason prompt engineering exists.


## Step 4. Prompt evaluation

Before we fix anything, we need a way to **measure**. Otherwise "better" is just a feeling.

An evaluation (an "eval") has three parts:

| Part | What it is |
|---|---|
| **Test cases** | A pile of realistic inputs, including nasty ones |
| **Run** | Feed each test case through your prompt |
| **Grader** | Score each output |

**Three kinds of grader:**

1. **Code grader**: a normal program checks the output. Fast, free, exact. Good for "is this valid JSON?" or "is the category one of my 5 allowed values?"
2. **Model grader**: another Claude reads the output and scores it. Good for judgement calls like "is this summary accurate?"
3. **Human grader**: you read it. Most accurate, slowest, most expensive. Use it on a small sample.

We will use a code grader **and** a model grader together, which is the normal setup.


### 4a. Generate test cases

Writing 20 realistic messy customer messages by hand is boring. Let Claude do it.

Use a **small, fast, cheap model** here (Haiku). Generating test data is a simple task, it does not need your most powerful model, and you usually want a lot of them.

Trick: ask for JSON only, and say explicitly "no markdown, no preamble". Then parse it.


In [ ]:
GENERATOR_PROMPT = """Generate 8 realistic customer support messages for an online clothing shop.

Make them varied and messy, like real people write:
- some with typos and no punctuation
- some very short ("wheres my order")
- some long and rambling with several problems in one message
- some angry, some polite
- at least one that is not really a support issue at all (spam or a random question)

Return ONLY a JSON array of strings. No markdown fences, no explanation."""

r = client.messages.create(
    model=SMALL_MODEL,
    max_tokens=2000,
    messages=[{"role": "user", "content": GENERATOR_PROMPT}],
)

raw = r.content[0].text.strip()
raw = raw.replace("```json", "").replace("```", "").strip()  # safety net
test_cases = json.loads(raw)

for i, t in enumerate(test_cases, 1):
    print(f"{i}. {t[:90]}...")

### 4b. The code grader

This one checks the **shape** of the output, not the meaning. Cheap and strict.

Our rules:
- It must be valid JSON
- It must have exactly the keys `category`, `urgency`, `summary`
- `category` must be one of our 5 allowed values
- `urgency` must be one of `low`, `medium`, `high`
- `summary` must be a short string

Score: 1 point per rule passed, out of 5.


In [ ]:
ALLOWED_CATEGORIES = ["shipping", "returns", "product_question", "billing", "other"]
ALLOWED_URGENCY = ["low", "medium", "high"]

def code_grader(output_text):
    score = 0
    notes = []

    try:
        data = json.loads(output_text.strip().replace("```json", "").replace("```", "").strip())
        score += 1
    except Exception:
        return 0, ["not valid JSON"]

    if set(data.keys()) == {"category", "urgency", "summary"}:
        score += 1
    else:
        notes.append(f"wrong keys: {list(data.keys())}")

    if data.get("category") in ALLOWED_CATEGORIES:
        score += 1
    else:
        notes.append(f"bad category: {data.get('category')}")

    if data.get("urgency") in ALLOWED_URGENCY:
        score += 1
    else:
        notes.append(f"bad urgency: {data.get('urgency')}")

    s = data.get("summary")
    if isinstance(s, str) and 0 < len(s) <= 120:
        score += 1
    else:
        notes.append("summary missing or too long")

    return score, notes

# quick sanity check
print(code_grader('{"category": "shipping", "urgency": "high", "summary": "Order delayed."}'))
print(code_grader('Sure! Here is the answer: this is a shipping issue.'))

### 4c. The model grader

The code grader cannot tell you if the summary is **accurate** or the urgency is **sensible**. For that we ask another Claude.

**The key trick (this is the quiz question 5 answer):** do not just ask for a number. Models hedge and give everything a 6 or 7. Ask for **strengths, weaknesses, and reasoning first**, then the score. Forcing it to think out loud spreads the scores out and makes them honest.

Note we also ask for JSON only so we can parse it.


In [ ]:
GRADER_SYSTEM = """You grade the quality of a support ticket classification.

You will be given the original customer message and the classifier's output.

First analyse, then score. Return ONLY this JSON, no markdown:
{
  "strengths": "what the classification got right",
  "weaknesses": "what it got wrong or missed",
  "reasoning": "weigh those up and justify the score",
  "score": <integer 1 to 10>
}

Scoring guide:
1-3  = category clearly wrong, or summary misrepresents the message
4-6  = roughly right but urgency misjudged or summary vague
7-8  = correct and useful, minor nitpicks
9-10 = correct, well judged urgency, summary a human could act on immediately"""

def model_grader(ticket, output_text):
    r = client.messages.create(
        model=BIG_MODEL,
        max_tokens=700,
        system=GRADER_SYSTEM,
        messages=[{"role": "user", "content":
            f"CUSTOMER MESSAGE:\n{ticket}\n\nCLASSIFIER OUTPUT:\n{output_text}"}],
    )
    txt = r.content[0].text.strip().replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(txt)
    except Exception:
        return {"strengths": "", "weaknesses": "grader returned unparseable output",
                "reasoning": txt[:200], "score": 0}

### 4d. Run the full evaluation on prompt v1

Now we put it together: every test case goes through the prompt, then through both graders.


In [ ]:
def run_eval(system_prompt, cases, label):
    results = []
    for ticket in cases:
        out = classify(ticket, system_prompt)
        c_score, c_notes = code_grader(out)
        m = model_grader(ticket, out)
        results.append({
            "ticket": ticket,
            "output": out,
            "code_score": c_score,
            "code_notes": c_notes,
            "model_score": m["score"],
            "weaknesses": m["weaknesses"],
        })

    avg_code = sum(r["code_score"] for r in results) / len(results)
    avg_model = sum(r["model_score"] for r in results) / len(results)
    print(f"=== {label} ===")
    print(f"Average code score : {avg_code:.2f} / 5")
    print(f"Average model score: {avg_model:.2f} / 10\n")
    return results, avg_code, avg_model

results_v1, code_v1, model_v1 = run_eval(PROMPT_V1, test_cases, "PROMPT V1")

for r in results_v1[:3]:
    print("TICKET :", r["ticket"][:70])
    print("OUTPUT :", r["output"][:120].replace("\n", " "))
    print("SCORES : code", r["code_score"], "| model", r["model_score"])
    print("ISSUES :", r["code_notes"], "|", r["weaknesses"][:80])
    print("-" * 70)

Expect a low code score, probably near 0 or 1. Prompt v1 never promised JSON, so it never delivers it.

**Now we know exactly what to fix.** That is the point of evaluating first.


## Step 5. Prompt engineering

Prompt engineering is just writing clearer instructions. The techniques that do most of the work:

| Technique | What it means |
|---|---|
| **Give a role** | "You are a support triage system" |
| **Be specific about the task** | Say exactly what to produce |
| **Constrain the options** | List the 5 allowed categories, do not let it invent |
| **Specify the exact format** | Show the JSON shape you want |
| **Handle edge cases** | Say what to do with spam, or messages with 3 problems |
| **Give examples (few shot)** | One or two worked examples, hugely effective |
| **Say what NOT to do** | "No markdown, no preamble" |

Watch how each of these maps onto a problem the eval just found.


In [ ]:
PROMPT_V2 = """You are a support ticket triage system for an online clothing shop.

TASK
Read one customer message and classify it. Output structured data only.

CATEGORY - choose exactly one:
- shipping        : delivery, tracking, lost or late parcels
- returns         : returns, refunds, exchanges, wrong or faulty item
- product_question: sizing, materials, stock, care instructions
- billing         : payments, charges, discount codes, invoices
- other           : anything else, including spam and off-topic messages

URGENCY - choose exactly one:
- high  : customer is blocked, has a hard deadline, or has been charged incorrectly
- medium: a real problem but no deadline mentioned
- low   : a question, or a message needing no action

SUMMARY
One sentence, under 120 characters, written for a support agent. State the problem and
any deadline. Do not copy the customer's words verbatim.

EDGE CASES
- If the message contains several problems, classify the most urgent one.
- If it is spam or not a support issue, use category "other" and urgency "low".
- If the message is too vague to place, use "other" and say so in the summary.

OUTPUT FORMAT
Return ONLY this JSON object. No markdown fences, no preamble, no explanation.
{"category": "...", "urgency": "...", "summary": "..."}

EXAMPLE
Message: "ordered a hoodie 3 weeks ago, still nothing, need it before my trip on the 12th"
Output: {"category": "shipping", "urgency": "high", "summary": "Hoodie not delivered after 3 weeks; customer needs it before the 12th."}"""

print(classify(TICKET, PROMPT_V2))

## Step 6. Re-run the evaluation and compare

Same test cases, same graders, new prompt. This is the only fair way to say "it got better".


In [ ]:
results_v2, code_v2, model_v2 = run_eval(PROMPT_V2, test_cases, "PROMPT V2")

print("COMPARISON")
print(f"Code grader : {code_v1:.2f} -> {code_v2:.2f}  (out of 5)")
print(f"Model grader: {model_v1:.2f} -> {model_v2:.2f}  (out of 10)")

In [ ]:
# Look at the worst remaining cases. This is where your next improvement comes from.
worst = sorted(results_v2, key=lambda r: (r["code_score"], r["model_score"]))[:3]

for r in worst:
    print("TICKET :", r["ticket"][:80])
    print("OUTPUT :", r["output"][:150].replace("\n", " "))
    print("SCORES : code", r["code_score"], "| model", r["model_score"])
    print("WEAKNESS:", r["weaknesses"][:150])
    print("-" * 70)

## The loop

You do not do this once. You do it in circles:

```
write prompt  ->  run eval  ->  read the worst failures  ->  edit prompt  ->  run eval again
        ^                                                                        |
        +------------------------------------------------------------------------+
```

Stop when the score is good enough for what you are building, not when it is perfect.


## Cheat sheet

| Term | Very simple meaning |
|---|---|
| **Claude API** | You send text to a URL, Claude sends text back. No memory between calls. |
| **System prompt** | The job description you give Claude. Set once, applies to every message. |
| **User message** | The specific task right now. |
| **Prompt engineering** | Writing clearer instructions: role, rules, allowed options, exact format, examples. |
| **Test case** | One realistic input you check your prompt against. Include the messy ones. |
| **Evaluation** | Running many test cases and scoring the results. |
| **Code grader** | A program checks the output. Fast, exact, only checks structure. |
| **Model grader** | Another Claude scores the output. Ask for strengths and weaknesses BEFORE the score. |
| **Human grader** | You read it. Best quality, use on a small sample. |

**The five things to actually remember:**

1. Testing a prompt once proves nothing. Real users break things.
2. Put the rules in the system prompt, the task in the user message.
3. Specify the exact output format, and list the allowed values.
4. Use a cheap fast model to generate test data.
5. Make your model grader explain itself before it scores.
